# ECON 3291 — Working with Data in R

### A techniques notebook

This notebook teaches the **methods** you need for the course. It deliberately uses
variables that the problem set does not ask about, so you can run every cell here and
still have to do your own thinking on the assignment.

**Before you run anything:** go to **Runtime → Change runtime type** and set the runtime
to **R**. Colab uses Python by default, and every cell below will fail on the Python runtime.

Work through it top to bottom. Change the code and re-run it — that is the point.


---
## 0. Setup


In [ ]:
library(dplyr)
library(ggplot2)

In [ ]:
# The dataset is published as a Google Sheet, so there is nothing to upload.
sheet_id <- "1RZxJHcFtfxGp8BiRQdA0YF6VKwT43MOY"
csv_url  <- paste0("https://docs.google.com/spreadsheets/d/", sheet_id, "/export?format=csv")

df <- read.csv(csv_url)

---
## 1. Know your data before you calculate

Nearly every analysis mistake is made before the first calculation. Three questions:

1. What does **one row** represent?
2. Which columns **change within a unit**, and which are fixed?
3. How many **distinct units** are there, as opposed to rows?


In [ ]:
str(df)     # how many rows, and what type is each column?

In [ ]:
head(df)    # what do the first few rows actually look like?

In [ ]:
names(df)   # exact column names - useful when R says a column does not exist

`table()` on a pair of columns is the fastest structural check there is. It reveals empty
cells, unexpected categories and coding mistakes in about a second.


In [ ]:
table(df$round)                 # is the panel balanced?
table(df$hhsize > 6, df$round)  # how a rule you invent splits the data


In [ ]:
table(df$round)

> **Try it:** this dataset has one row per household *per round*. Sort mentally by `hhid`
> and look at `head(df)` again — you should see each household appear twice.


---
## 2. Describing a variable

One function per statistic. We will use `agehh` (age of the household head) throughout this
section, which is not what the problem set asks about.


In [ ]:
mean(df$agehh)
median(df$agehh)
sd(df$agehh)

`sd()` is the **sample** standard deviation — the same choice as Excel's `STDEV.S`.


In [ ]:
min(df$agehh)
max(df$agehh)
range(df$agehh)   # both at once

**Always run `min` and `max` first.** An age of 0 or 900, or a negative expenditure, tells
you something is wrong before you waste an hour on the analysis.


### The whole table in one call

`sapply()` applies a function to each column you name and assembles the results.


In [ ]:
vars <- c("agehh", "hhsize")

sapply(df[vars], function(x) c(Mean = mean(x),
                               SD   = sd(x),
                               Min  = min(x),
                               Max  = max(x)))

> **Try it:** add `"hhsize"` to `vars` and re-run. You did not have to write a single new
> line per statistic — that is the point of doing it this way.


---
## 3. Counting: rows are not entities

This is the single most important distinction in the course. In a panel, counting rows counts
every unit once per period.


In [ ]:
nrow(df)                    # rows

In [ ]:
length(unique(df$hhid))     # distinct households

Those two numbers are different, and **which one is correct depends on the question**.
“How many observations?” wants the first. “How many households?” wants the second.


### Counting things that meet a condition


In [ ]:
sum(df$hhsize > 6)          # how many ROWS have more than six members
mean(df$hhsize > 6)          # what PROPORTION of rows do

A logical condition is `TRUE`/`FALSE`, which R treats as 1/0. So `sum()` counts and
`mean()` gives the proportion. That one fact replaces a lot of arithmetic.


> **Try it:** count the distinct households with more than six members, rather than the rows.
> You will need to combine `unique()` with a subset — see the next section.


---
## 4. Subsetting: choosing the rows you mean

Square brackets take `[rows, columns]`. Leave a side blank to keep all of it.


In [ ]:
base <- df[df$round == 0, ]     # one period only
nrow(base)

In [ ]:
nrow(df[!duplicated(df$hhid), ])   # the first row for each household


Naming a subset once and reusing it beats repeating the filter everywhere. If the definition
changes later, you edit one line instead of hunting for every place you wrote it.


In [ ]:
big <- df[df$hhsize > 6, ]
nrow(big)
length(unique(big$hhid))

---
## 5. Creating a variable from a rule

`ifelse()` reads as a sentence: if the condition holds, then this, otherwise that.


In [ ]:
df$big <- ifelse(df$hhsize > 6, 1, 0)

mean(df$big)   # because it is 1s and 0s, the mean IS the proportion

In [ ]:
df$logexp   <- log(df$hhe)
df$agegroup <- ifelse(df$agehh >= 60, "older", "younger")

table(df$agegroup)

The assignment arrow creates the column if it does not exist and overwrites it if it does, so
running the same cell twice is harmless — which is exactly what you want in a notebook.


---
## 6. Comparing groups

The `%>%` operator passes the result on its left into the function on its right. Read a
pipeline top to bottom as a sentence.


In [ ]:
df$big <- ifelse(df$hhsize > 6, 1, 0)   # a grouping of your own making

df %>%
  filter(round == 0) %>%              # keep the rows you want
  group_by(big) %>%                   # split into groups
  summarise(mean_age = mean(agehh),
            n        = n())           # ALWAYS report the group size


**A mean with no `n` beside it is not reportable.** Include `n = n()` every time.


> **Try it:** group by `agegroup` instead of `big`, and add the median household size.
>
> Note: do **not** group by `local` here. In this dataset locality and treatment status
> are the same split, so grouping by it answers a problem-set question for you.


---
## 7. Plotting

`ggplot` builds a plot in layers, joined with `+`: the data, then what to draw, then how to
label it.


In [ ]:
ggplot(base, aes(x = agehh, y = hhe)) +
  geom_point(color = "steelblue") +
  labs(x = "Age of household head",
       y = "Health expenditure") +
  theme_minimal()

- `aes()` maps variables to axes.
- each `geom_` adds a layer of marks.
- `labs()` labels the axes. **Do this every time** — a plot with bare variable names is not finished.


> **Try it:** add `+ geom_smooth(method = "lm")` to draw a fitted line, or put
> `colour = factor(treatcom)` inside `aes()` to split the points by group.


---
## 8. Correlation

`cor()` returns a number between −1 and +1. It measures **direction and tightness only** —
not slope size, and not cause.


In [ ]:
cor(base$agehh, base$hhe)

Notice that the variable names are *in the line*. Anyone reading this can see exactly what was
correlated.

The Excel equivalent is less self-describing. `CORREL` has no "IFS" form, so you would first have
to copy the baseline rows onto their own sheet — the rows alternate between round 1 and round 0,
so you cannot just take the top block — and then point at two lettered columns:
`=CORREL(F2:F276, D2:D276)`. Nothing on screen tells you that F is `agehh` and D is `hhe`, or that
the sheet really is baseline-only.

**Plot before you compute.** A correlation near zero can hide a strong curved relationship, and
a large one can be produced by a handful of outliers.


> **Try it:** compute the same correlation on the full `df` rather than `base`. Does restricting
> to one round change it? Whichever you report, say which subset you used.


---
## 9. Regression

The `~` means *explained by*. Add regressors with `+`.


In [ ]:
m1 <- lm(hhe ~ agehh, data = base)
summary(m1)

In [ ]:
m2 <- lm(hhe ~ agehh + hhsize, data = base)
summary(m2)

Store the model in an object, then ask it questions. You fit once and can extract whatever you
need afterwards without refitting.


In [ ]:
coef(m2)      # just the coefficients
confint(m2)   # confidence intervals

Compare `summary()` with Excel's `LINEST` block: named rows, in the order you wrote them, with
t-statistics and p-values already computed. No counting across a grid, and no reading backwards.


---
## 10. Your turn

None of these are on the problem set. Write the code yourself in the empty cells.


**(a)** Describe `hhsize`: mean, median, standard deviation and range.


**(b)** Count the rows, then the distinct households. Explain the difference in a text cell.


**(c)** Build an indicator for household heads aged 50 or over, and report the proportion.


**(d)** Compare mean household size at baseline across your `big` groups, reporting `n`
for each. Then say in a text cell why the `n` matters.


**(e)** Plot `hhsize` against `hhe`, then fit `lm(hhe ~ hhsize)` and read the output.


---
## Where to go next

- Take these techniques to the problem set. The methods are identical; only the
  variables change, and working out *which* variables is part of the exercise.
- When something breaks, check the four usual causes: wrong runtime, a package not loaded,
  a cell run out of order, or a misspelled column name.
- `Runtime → Restart and run all` fixes most out-of-order problems and costs thirty seconds.

Prepared by **Sean Rogers**, DITI Research and Teaching Fellows.

Questions: **nulab.info@gmail.com** · Book time with us: **https://bit.ly/diti-meeting**
